# Savant 00, baseline before any training

This notebook measures the untrained base model inside the Savant harness, so later training has an honest starting point. It runs on the held-out bench tables, which are never used for tuning.

1. The guarded statistics engine runs alone on every table, which needs no model.
2. The base model then answers one open question per table twice. In blind mode it sees only the data card and must find patterns with its own code. In guided mode it also sees the engine's guarded findings.
3. Each answer ends with PAIR lines, which are graded against the planted truth exactly like the engine, so recall and false findings are comparable.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import json
import subprocess
import sys
import time
from pathlib import Path

build = sorted(glob.glob("/kaggle/input/**/BUILD", recursive=True))
assert build and Path(build[0]).read_text().strip() == "EXPECTED_BUILD", "stale bench dataset, stop"
wheel = sorted(glob.glob("/kaggle/input/**/savant-*.whl", recursive=True))[-1]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel], check=True)
TASKS = sorted(glob.glob("/kaggle/input/**/tasks/*.csv", recursive=True))
print(wheel, len(TASKS), "tables")

In [ ]:
# Settings for this run. The model can be swapped for Qwen/Qwen3.5-4B once transformers on Kaggle supports it.
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
TABLES = {"null": 3, "hard-null": 3, "plant": 4}  # tables per kind given to the model
MODES = ["blind", "guided"]
QUESTION = (
    "What are the most important relationships between columns in this data? "
    "Report only what the data supports. End your answer with one line per relationship "
    "in the form PAIR: column_a | column_b, or the single line PAIR: none."
)
MAX_NEW_TOKENS = 768  # longest single model reply
KEEP_MESSAGES = 6  # recent messages the model sees each round

In [ ]:
from pathlib import Path

import polars as pl

from savant.engine import Settings, analyze
from savant.grade import grade, summarize
from savant.planted import Plant, Task


def load_task(csv):
    truth = json.loads(Path(csv.replace(".csv", ".truth.json")).read_text())
    plants = [Plant(p["kind"], tuple(p["columns"]), p["strength"]) for p in truth["plants"]]
    return Task(truth["id"], pl.read_csv(csv, try_parse_dates=True), plants)


tasks = [load_task(c) for c in TASKS]
t0 = time.time()
engine_scores = [grade(t, analyze(t.table, Settings(stability_runs=100))) for t in tasks]
engine_summary = summarize(engine_scores)
print(f"{time.time() - t0:.0f}s")
print(json.dumps(engine_summary, indent=1))

In [ ]:
import torch

from savant.agent import ask, parse_pairs
from savant.grade import derived_pairs
from savant.llm import Transformers

client = Transformers(MODEL_ID, max_new_tokens=MAX_NEW_TOKENS)
chosen = []
for prefix, count in TABLES.items():
    chosen += [t for t in tasks if t.id.startswith(prefix + "-")][:count]
log = Path("/kaggle/working/runs.jsonl")
log.write_text("")
runs = []
for mode in MODES:
    for t in chosen:
        t0 = time.time()
        try:
            r = ask(
                t.table,
                QUESTION,
                client,
                keep_messages=KEEP_MESSAGES,
                show_findings=(mode == "guided"),
            )
            error = ""
        except torch.cuda.OutOfMemoryError as e:
            r, error = None, f"out of memory: {e}"[:300]
        torch.cuda.empty_cache()
        real = {frozenset(p.columns) for p in t.plants if p.kind != "leak"}
        leaks = {frozenset(p.columns) for p in t.plants if p.kind == "leak"}
        claimed = parse_pairs(r.rendered if r else "", t.table.columns)
        row = {
            "mode": mode,
            "task": t.id,
            "kind": t.id.rsplit("-", 1)[0],
            "grounded": bool(r and r.grounded),
            "rounds": r.rounds if r else None,
            "steps": len(r.steps) if r else 0,
            "failed_steps": sum(not s.ok for s in r.steps) if r else 0,
            "planted": len(real),
            "found": len(claimed & real),
            "false": len(claimed - real - leaks - derived_pairs(t)),
            "seconds": round(time.time() - t0, 1),
            "error": error,
            "answer": r.rendered if r else None,
            "transcript": r.transcript if r else None,
        }
        runs.append(row)
        with log.open("a") as f:
            f.write(json.dumps(row, default=str) + "\n")
        print(
            mode,
            t.id,
            "grounded" if row["grounded"] else "NOT grounded",
            row["rounds"],
            "rounds",
            row["found"],
            "of",
            row["planted"],
            "found",
            row["false"],
            "false",
            row["seconds"],
            "s",
            error,
        )

In [ ]:
def share(xs):
    return round(sum(xs) / len(xs), 3) if xs else None


agent_summary = {"model": MODEL_ID}
for mode in MODES:
    rs = [r for r in runs if r["mode"] == mode]
    planted = [r for r in rs if r["kind"] == "plant"]
    agent_summary[mode] = {
        "tables": len(rs),
        "grounded_answers": share([r["grounded"] for r in rs]),
        "mean_rounds": share([r["rounds"] for r in rs if r["rounds"]]),
        "mean_seconds": share([r["seconds"] for r in rs]),
        "failed_step_share": round(
            sum(r["failed_steps"] for r in rs) / max(1, sum(r["steps"] for r in rs)), 3
        ),
        "recall": round(
            sum(r["found"] for r in planted) / max(1, sum(r["planted"] for r in planted)), 3
        ),
        "false_per_null_table": share([r["false"] for r in rs if r["kind"] == "null"]),
        "false_per_hard_null_table": share([r["false"] for r in rs if r["kind"] == "hard-null"]),
        "false_per_planted_table": share([r["false"] for r in planted]),
        "errors": sum(bool(r["error"]) for r in rs),
    }
Path("/kaggle/working/baseline.json").write_text(
    json.dumps({"engine": engine_summary, "agent": agent_summary}, indent=1, default=str)
)
print(json.dumps(agent_summary, indent=1))